# 🐾 Animal Classification — MaxViT-Base/256 (Head-Only)

**5-Fold CV · Optuna HPO · Macro F1 · Frozen→Finetune · LLRD**

**Run `00_shared_splits` first.**

- MaxViT-Base @ 256px — backbone permanently frozen, only head trains.
- - `maxvit_base_tf_256` in timm — 120M params, strongest MaxViT variant
- - Backbone NEVER unfreezes — pure linear probe approach
- - Dual avg+max pool head over spatial feature map
- - Head-only training: higher lr (1e-3), no LLRD needed
- - Fast convergence: 20 epochs typically sufficient

## 0. Config

In [ ]:
SPLIT_SEED   = 42
N_FOLDS      = 5
SPLITS_DIR   = "/kaggle/working/splits"

CFG = dict(
    model_name    = "maxvit_base",
    img_size      = 256,
    batch_size    = 32,
    num_workers   = 2,
    warmup_epochs = 0,            # no warmup — head-only from epoch 1
    total_epochs  = 25,
    patience      = 7,
    lr            = 1e-3,         # high lr fine — backbone frozen
    wd            = 1e-4,
    lr_decay      = 1.0,          # unused — head-only
    scheduler     = "cosine",
    dropout       = 0.3,
    hidden_dim    = 512,
    mixup_alpha   = 0.0,          # no mixup for head-only (simpler)
    label_smooth  = 0.1,
    grad_clip     = 1.0,
    n_trials      = 20,
    output_dir    = "/kaggle/working",
)
OPTUNA_WARMUP_EP  = 0
OPTUNA_FT_EP      = 4
OPTUNA_BS_CHOICES = [16, 32, 64]


## 1. Imports & Setup

In [ ]:
import os, sys, json, logging, zipfile, random, time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import f1_score

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SPLIT_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = device.type == "cuda"
print("Device:", device)

OUT     = Path(CFG["output_dir"])
WEIGHTS = OUT / "weights"; LOGS = OUT / "logs"; SUBS = OUT / "submissions"
for d in [WEIGHTS, LOGS, SUBS]: d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOGS / "training.log"),
        logging.StreamHandler(sys.stdout),
    ]
)
log = logging.getLogger(__name__)
log.info(f"Config: {CFG}")


## 2. Load Data & Splits

In [ ]:
def find_file(name):
    for root, _, files in os.walk("/kaggle/input"):
        if name in files: return os.path.join(root, name)
    raise FileNotFoundError(name)

train_csv    = find_file("train.csv")
sample_csv   = find_file("sample_submission.csv")
base_dir     = os.path.dirname(train_csv)
TRAIN_DIR    = os.path.join(base_dir, "train_images")
TEST_DIR     = os.path.join(base_dir, "test_images")

full_df       = pd.read_csv(train_csv)
submission_df = pd.read_csv(sample_csv)

classes      = sorted(full_df["animal_class"].unique())
NUM_CLASSES  = len(classes)
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
full_df["label"] = full_df["animal_class"].map(class_to_idx)

holdout_data = np.load(f"{SPLITS_DIR}/holdout.npz")
holdout_idx  = holdout_data["holdout_idx"]
trainval_idx = holdout_data["trainval_idx"]

log.info(f"Total:{len(full_df)} TrainVal:{len(trainval_idx)} Holdout:{len(holdout_idx)} Classes:{NUM_CLASSES}")
print(full_df.head(3))


## 3. Augmentation & Datasets

In [ ]:
# ── Core idea: colour is NOT a class signal in this competition ──
# Segmentation images have arbitrary colours per instance.
# Strategy: destroy colour information aggressively, force the model
# to learn SHAPE, SILHOUETTE and STRUCTURAL cues only.

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
SZ   = CFG["img_size"]

# Strong colour destruction + shape-preserving spatial augmentation
train_tf = transforms.Compose([
    transforms.Resize((int(SZ * 1.15), int(SZ * 1.15))),   # slightly oversized then crop
    transforms.RandomCrop((SZ, SZ)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(degrees=30),
    # Colour destruction — hue=0.5 is max, full saturation/brightness swing
    transforms.ColorJitter(brightness=0.8, contrast=0.8, saturation=1.0, hue=0.5),
    # 30 % chance of full grayscale — forces shape-only features
    transforms.RandomGrayscale(p=0.3),
    # Simulate different segmentation renders
    transforms.RandomPosterize(bits=4, p=0.2),
    transforms.RandomSolarize(threshold=128, p=0.1),
    transforms.RandomAffine(
        degrees=0, translate=(0.15, 0.15),
        scale=(0.8, 1.2), shear=10,
    ),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    # Random erasing: occludes body parts, prevents relying on specific regions
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2), ratio=(0.3, 3.3)),
])

val_tf = transforms.Compose([
    transforms.Resize((SZ, SZ)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# TTA transform — used at inference only (slight flips + colour jitter)
def tta_transforms(img_size):
    return [
        transforms.Compose([transforms.Resize((img_size, img_size)),
                             transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
        transforms.Compose([transforms.Resize((img_size, img_size)),
                             transforms.RandomHorizontalFlip(p=1.0),
                             transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
        transforms.Compose([transforms.Resize((int(img_size*1.1), int(img_size*1.1))),
                             transforms.CenterCrop(img_size),
                             transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    ]
TTA_TFS = tta_transforms(SZ)


class AnimalDS(Dataset):
    def __init__(self, df, image_dir, tf=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir; self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(os.path.join(self.image_dir, row["image_id"])).convert("RGB")
        if self.tf: img = self.tf(img)
        return img, int(row["label"])

class TestDS(Dataset):
    def __init__(self, df, image_dir, tf):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir; self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        iid = self.df.iloc[i]["image_id"]
        return self.tf(Image.open(os.path.join(self.image_dir, iid)).convert("RGB")), iid

test_ds   = TestDS(submission_df, TEST_DIR, val_tf)
te_loader = DataLoader(test_ds, batch_size=CFG["batch_size"],
                       shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)
print("Datasets & TTA transforms ready.")


## 4. Training Helpers

In [ ]:
# ── Mixup ─────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return x * lam + x[idx] * (1 - lam), y, y[idx], lam

def mixup_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)


def make_loaders(tr_df, val_df, bs=None):
    bs = bs or CFG["batch_size"]
    return (
        DataLoader(AnimalDS(tr_df, TRAIN_DIR, train_tf),
                   batch_size=bs, shuffle=True,
                   num_workers=CFG["num_workers"], pin_memory=True, drop_last=True),
        DataLoader(AnimalDS(val_df, TRAIN_DIR, val_tf),
                   batch_size=bs, shuffle=False,
                   num_workers=CFG["num_workers"], pin_memory=True),
    )


def train_epoch(model, loader, optimizer, criterion, scaler, use_mixup=True):
    model.train()
    tl, correct, total = 0., 0, 0
    preds_all, labels_all = [], []
    n_steps = len(loader)

    for step, (imgs, labels) in enumerate(loader, 1):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        # Mixup (disabled during warmup via use_mixup flag)
        if use_mixup and CFG.get("mixup_alpha", 0) > 0:
            imgs, ya, yb, lam = mixup_data(imgs, labels, CFG["mixup_alpha"])
            with torch.cuda.amp.autocast(enabled=AMP):
                out  = model(imgs)
                loss = mixup_criterion(criterion, out, ya, yb, lam)
        else:
            with torch.cuda.amp.autocast(enabled=AMP):
                out  = model(imgs)
                loss = criterion(out, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.get("grad_clip", 1.0))
        scaler.step(optimizer); scaler.update()

        tl += loss.item() * imgs.size(0)
        preds = out.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

        if step % 20 == 0:
            run_f1   = f1_score(labels_all, preds_all, average="macro", zero_division=0)
            run_acc  = correct / total
            run_loss = tl / total
            log.info(
                f"    step {step:04d}/{n_steps} | "
                f"loss={run_loss:.4f}  acc={run_acc:.4f}  macro_f1={run_f1:.4f}"
            )

    return tl/total, correct/total, f1_score(labels_all, preds_all, average="macro", zero_division=0)


@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    tl, correct, total = 0., 0, 0
    preds_all, labels_all, probs_all = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.cuda.amp.autocast(enabled=AMP):
            out  = model(imgs)
            loss = criterion(out, labels)
        tl += loss.item() * imgs.size(0)
        preds = out.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
        probs_all.append(F.softmax(out, dim=1).cpu().numpy())
    val_f1 = f1_score(labels_all, preds_all, average="macro", zero_division=0)
    return tl/total, correct/total, val_f1, np.concatenate(probs_all)


@torch.no_grad()
def predict_tta(model, df, image_dir):
    """Test-time augmentation: average softmax over TTA_TFS transforms."""
    model.eval()
    all_probs = []
    for tf in TTA_TFS:
        ds     = TestDS(df, image_dir, tf)
        loader = DataLoader(ds, batch_size=CFG["batch_size"], shuffle=False,
                            num_workers=CFG["num_workers"], pin_memory=True)
        probs  = []
        for imgs, _ in loader:
            with torch.cuda.amp.autocast(enabled=AMP):
                probs.append(F.softmax(model(imgs.to(device)), dim=1).cpu().numpy())
        all_probs.append(np.concatenate(probs))
    return np.mean(all_probs, axis=0)   # (N, C) averaged over TTA views


def make_scheduler(optimizer, name, epochs):
    if name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-8)
    elif name == "cosine_wr":
        return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=max(1,epochs//3), T_mult=2)
    else:
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1,epochs//3), gamma=0.3)


print("Helpers ready  (mixup + TTA enabled).")


## 5. Plotting Helpers

In [ ]:
def plot_fold(fold, h):
    epochs = range(1, len(h["tr_loss"])+1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f"{CFG['model_name']}  |  Fold {fold}  |  best val F1={max(h['val_f1']):.4f}", fontsize=12)
    for ax, (tk, vk, title, yl) in zip(axes, [
        ("tr_loss","val_loss","Loss","Loss"),
        ("tr_f1",  "val_f1",  "Macro F1","F1"),
        ("tr_acc", "val_acc", "Accuracy","Acc"),
    ]):
        ax.plot(epochs, h[tk], label="train", marker="o", ms=3)
        ax.plot(epochs, h[vk], label="val",   marker="o", ms=3)
        if "wu_end" in h:
            ax.axvline(x=h["wu_end"]+.5, color="gray", ls="--", alpha=.5, label="unfreeze")
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(yl)
        ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show(); plt.close()

def plot_optuna(study):
    try:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        trials = [t for t in study.trials if t.value is not None]
        vals   = [t.value for t in trials]
        best_so_far = [max(vals[:i+1]) for i in range(len(vals))]
        axes[0].plot(vals, marker="o", ms=4, alpha=0.6, label="trial F1")
        axes[0].plot(best_so_far, color="red", lw=2, label="best so far")
        axes[0].set_title("Optuna — optimisation history")
        axes[0].set_xlabel("Trial"); axes[0].set_ylabel("Val macro F1")
        axes[0].legend(); axes[0].grid(alpha=.3)
        try:
            imp = optuna.importance.get_param_importances(study)
            axes[1].barh(list(imp.keys()), list(imp.values()))
            axes[1].set_title("Param importances"); axes[1].set_xlabel("Importance")
            axes[1].invert_yaxis(); axes[1].grid(alpha=.3)
        except Exception:
            axes[1].text(0.5, 0.5, "importance N/A
(need ≥2 trials)", ha="center", va="center")
        plt.tight_layout(); plt.show(); plt.close()
    except Exception as e:
        print(f"Optuna plot skipped: {e}")


## 6. Model — MaxViT-Base/256 (Head-Only)

In [ ]:
import timm
__import__("subprocess").run(["pip","install","-q","--upgrade","timm"])

class MaxViTBaseClassifier(nn.Module):
    """
    MaxViT-Base @ 256px — backbone PERMANENTLY FROZEN.
    Only the classification head trains.
    Dual avg+max pool over spatial feature map.
    """
    def __init__(self, dropout=0.3, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            "maxvit_base_tf_256",
            pretrained=True,
            num_classes=0,
            global_pool="",
        )
        # Freeze permanently inside __init__
        for p in self.backbone.parameters():
            p.requires_grad = False

        in_f = self.backbone.num_features   # 1024 for maxvit_base
        self.head = nn.Sequential(
            nn.LayerNorm(in_f * 2),
            nn.Linear(in_f * 2, hidden_dim), nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, NUM_CLASSES),
        )
    def forward(self, x):
        feat = self.backbone.forward_features(x)   # (B, H, W, C)
        if feat.dim() == 4: feat = feat.permute(0, 3, 1, 2)
        avg = F.adaptive_avg_pool2d(feat, 1).flatten(1)
        mx  = F.adaptive_max_pool2d(feat, 1).flatten(1)
        return self.head(torch.cat([avg, mx], dim=1))

def build_model(dropout=None, hidden_dim=None):
    return MaxViTBaseClassifier(
        dropout    = dropout    or CFG["dropout"],
        hidden_dim = hidden_dim or CFG["hidden_dim"],
    )
# Backbone is already frozen inside __init__ — these are no-ops but
# kept for compatibility with the shared training loop
def freeze_backbone(m): pass
def unfreeze_backbone(m): pass   # ← never called; warmup_epochs=0
def get_head_params(m): return list(m.head.parameters())
def build_optimizer(m, lr=None, wd=None, lr_decay=None):
    # Only head params — backbone has requires_grad=False
    lr = lr or CFG["lr"]; wd = wd or CFG["wd"]
    return torch.optim.AdamW(m.head.parameters(), lr=lr, weight_decay=wd)

# Verify backbone is truly frozen
_m = build_model().to(device)
trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
head_only  = sum(p.numel() for p in _m.head.parameters())
assert trainable == head_only, f"Backbone leak! trainable={trainable} head={head_only}"
with torch.no_grad():
    assert _m(torch.randn(2,3,CFG["img_size"],CFG["img_size"]).to(device)).shape==(2,NUM_CLASSES)
in_f = _m.backbone.num_features
del _m; torch.cuda.empty_cache()
print(f"MaxViT-Base/256 OK  in_features={in_f}  trainable={head_only/1e3:.1f}K (head only)")


## 7. Optuna HPO

Runs on Fold 0 as proxy.  
Tunes: `lr`, `wd`, `dropout`, `hidden_dim`, `batch_size`, `scheduler`, `lr_decay`, `mixup_alpha`, `label_smooth`.

In [ ]:
_opt_data  = np.load(f"{SPLITS_DIR}/fold_0.npz")
_opt_tr_df = full_df.iloc[_opt_data["train_idx"]]
_opt_val_df= full_df.iloc[_opt_data["val_idx"]]

def _optuna_objective(trial):
    lr          = trial.suggest_float("lr",          1e-5, 3e-4, log=True)
    wd          = trial.suggest_float("wd",          1e-5, 1e-2, log=True)
    dropout     = trial.suggest_float("dropout",     0.1,  0.4)
    hidden_dim  = trial.suggest_categorical("hidden_dim",  [384, 512, 768])
    bs          = trial.suggest_categorical("batch_size",  OPTUNA_BS_CHOICES)
    sched       = trial.suggest_categorical("scheduler",   ["cosine", "cosine_wr"])
    lr_decay    = trial.suggest_float("lr_decay",    0.55, 0.85)
    mixup_alpha = trial.suggest_float("mixup_alpha", 0.0,  0.5)
    label_smooth= trial.suggest_float("label_smooth",0.05, 0.2)

    model     = build_model(dropout=dropout, hidden_dim=hidden_dim).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)
    scaler    = torch.cuda.amp.GradScaler(enabled=AMP)
    tr_loader, val_loader = make_loaders(_opt_tr_df, _opt_val_df, bs=bs)

    # Warmup — frozen, no mixup
    freeze_backbone(model)
    opt_wu = torch.optim.AdamW(get_head_params(model), lr=lr*10, weight_decay=wd)
    sch_wu = torch.optim.lr_scheduler.CosineAnnealingLR(opt_wu, T_max=OPTUNA_WARMUP_EP, eta_min=lr)
    for _ in range(OPTUNA_WARMUP_EP):
        train_epoch(model, tr_loader, opt_wu, criterion, scaler, use_mixup=False)
        sch_wu.step()

    # Finetune — unfrozen, with mixup
    unfreeze_backbone(model)
    _tmp_alpha = CFG.get("mixup_alpha", 0)
    CFG["mixup_alpha"] = mixup_alpha
    opt_ft = build_optimizer(model, lr=lr, wd=wd, lr_decay=lr_decay)
    sch_ft = make_scheduler(opt_ft, sched, OPTUNA_FT_EP)
    for ep in range(OPTUNA_FT_EP):
        train_epoch(model, tr_loader, opt_ft, criterion, scaler, use_mixup=True)
        sch_ft.step()
        _, _, vf1, _ = val_epoch(model, val_loader, criterion)
        trial.report(vf1, ep)
        if trial.should_prune():
            CFG["mixup_alpha"] = _tmp_alpha
            del model; torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()
    CFG["mixup_alpha"] = _tmp_alpha

    _, _, vf1, _ = val_epoch(model, val_loader, criterion)
    del model; torch.cuda.empty_cache()
    return vf1

log.info(f"Optuna: {CFG['n_trials']} trials | warmup={OPTUNA_WARMUP_EP} | finetune={OPTUNA_FT_EP}")
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SPLIT_SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study.optimize(_optuna_objective, n_trials=CFG["n_trials"], show_progress_bar=True)

BEST = study.best_params
log.info(f"Optuna best val macro F1 = {study.best_value:.4f}")
log.info(f"Best params: {BEST}")

pd.DataFrame([
    {"trial": t.number, "value": t.value, **t.params}
    for t in study.trials if t.value is not None
]).to_csv(LOGS / "optuna_trials.csv", index=False)

CFG.update({
    "lr":           BEST["lr"],
    "wd":           BEST["wd"],
    "dropout":      BEST["dropout"],
    "hidden_dim":   BEST["hidden_dim"],
    "batch_size":   BEST["batch_size"],
    "scheduler":    BEST["scheduler"],
    "lr_decay":     BEST["lr_decay"],
    "mixup_alpha":  BEST["mixup_alpha"],
    "label_smooth": BEST["label_smooth"],
})
log.info(f"CFG patched with best params: {CFG}")
plot_optuna(study)
print(f"Best params: {BEST}")


## 8. 5-Fold Training Loop

In [ ]:
oof_probs  = np.zeros((len(full_df), NUM_CLASSES))
test_probs = np.zeros((len(submission_df), NUM_CLASSES))
fold_scores = []

for fold in range(N_FOLDS):
    log.info(f"{'='*72}")
    log.info(
        f"FOLD {fold+1}/{N_FOLDS}  |  lr={CFG['lr']:.2e}  wd={CFG['wd']:.2e}  "
        f"bs={CFG['batch_size']}  sched={CFG['scheduler']}  "
        f"lr_decay={CFG['lr_decay']:.2f}  mixup={CFG['mixup_alpha']:.2f}"
    )
    log.info(f"{'='*72}")

    data = np.load(f"{SPLITS_DIR}/fold_{fold}.npz")
    tr_loader, val_loader = make_loaders(
        full_df.iloc[data["train_idx"]], full_df.iloc[data["val_idx"]])

    model     = build_model(dropout=CFG["dropout"], hidden_dim=CFG["hidden_dim"]).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.get("label_smooth", 0.1))
    scaler    = torch.cuda.amp.GradScaler(enabled=AMP)

    best_f1 = -1.; best_wt = WEIGHTS / f"fold{fold}_best.pt"
    no_imp  = 0
    history = dict(tr_loss=[], val_loss=[], tr_f1=[], val_f1=[], tr_acc=[], val_acc=[])
    phase   = "warmup"

    # Phase 1 — frozen backbone, no mixup
    freeze_backbone(model)
    opt_wu = torch.optim.AdamW(get_head_params(model),
                                lr=CFG["lr"]*10, weight_decay=CFG["wd"])
    sch_wu = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_wu, T_max=CFG["warmup_epochs"], eta_min=CFG["lr"])
    log.info(f"  Phase 1 — FROZEN | warmup_epochs={CFG['warmup_epochs']} | head lr={CFG['lr']*10:.2e}")

    for epoch in range(1, CFG["total_epochs"]+1):

        if epoch == CFG["warmup_epochs"] + 1:
            phase = "finetune"
            unfreeze_backbone(model)
            opt_ft = build_optimizer(model, lr=CFG["lr"],
                                      wd=CFG["wd"], lr_decay=CFG["lr_decay"])
            ft_ep  = CFG["total_epochs"] - CFG["warmup_epochs"]
            sch_ft = make_scheduler(opt_ft, CFG["scheduler"], ft_ep)
            log.info(
                f"  Phase 2 — UNFROZEN | lr_decay={CFG['lr_decay']:.2f} | "
                f"sched={CFG['scheduler']} | mixup_alpha={CFG['mixup_alpha']:.2f}"
            )

        optimizer  = opt_wu if phase == "warmup" else opt_ft
        scheduler  = sch_wu if phase == "warmup" else sch_ft
        use_mixup  = (phase == "finetune")

        t0 = time.time()
        trl, tra, trf = train_epoch(model, tr_loader, optimizer, criterion, scaler, use_mixup=use_mixup)
        vll, vla, vlf, vprobs = val_epoch(model, val_loader, criterion)
        scheduler.step()
        elapsed = time.time() - t0

        history["tr_loss"].append(trl);  history["val_loss"].append(vll)
        history["tr_f1"].append(trf);    history["val_f1"].append(vlf)
        history["tr_acc"].append(tra);   history["val_acc"].append(vla)

        if vlf > best_f1:
            best_f1 = vlf
            torch.save(model.state_dict(), best_wt)
            best_vprobs = vprobs.copy(); no_imp = 0; flag = "★ best"
        else:
            no_imp += 1; flag = f"patience {no_imp}/{CFG['patience']}"

        log.info(
            f"[{phase:8s}] Fold {fold+1} Ep {epoch:03d}/{CFG['total_epochs']} | "
            f"TRAIN → loss={trl:.4f}  acc={tra:.4f}  macro_f1={trf:.4f} | "
            f"VAL   → loss={vll:.4f}  acc={vla:.4f}  macro_f1={vlf:.4f} | "
            f"{elapsed:.0f}s | {flag}"
        )
        print(
            f"  [{phase:8s}] ep={epoch:03d}/{CFG['total_epochs']}  "
            f"tr_loss={trl:.4f}  tr_f1={trf:.4f}  tr_acc={tra:.4f}  |  "
            f"val_loss={vll:.4f}  val_f1={vlf:.4f}  val_acc={vla:.4f}  "
            f"[{elapsed:.0f}s] {flag}"
        )
        if phase == "finetune" and no_imp >= CFG["patience"]:
            log.info(f"  Early stop at epoch {epoch}"); break

    history["wu_end"] = CFG["warmup_epochs"]
    log.info(f"{'─'*72}")
    log.info(f"Fold {fold+1} DONE | best val macro F1 = {best_f1:.4f} | epochs run = {len(history['tr_f1'])}")
    log.info(f"{'─'*72}")
    print(f"\n  ✓ Fold {fold+1} best val macro F1 = {best_f1:.4f}\n")

    plot_fold(fold, history)
    fold_scores.append(best_f1)
    oof_probs[data["val_idx"]] = best_vprobs

    # TTA inference for this fold
    log.info(f"  Running TTA inference for fold {fold+1}...")
    model.load_state_dict(torch.load(best_wt, map_location=device))
    test_probs += predict_tta(model, submission_df, TEST_DIR)
    del model; torch.cuda.empty_cache()

test_probs /= N_FOLDS
log.info(f"{'='*72}")
log.info(f"ALL FOLDS DONE")
log.info(f"  Per-fold val macro F1 : {[round(s,4) for s in fold_scores]}")
log.info(f"  Mean CV macro F1      : {np.mean(fold_scores):.4f}")
log.info(f"  Std                   : {np.std(fold_scores):.4f}")
log.info(f"{'='*72}")
print(f"\nCV macro F1 : {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"Per-fold     : {[round(s,4) for s in fold_scores]}")


## 9. OOF Evaluation & Submission

In [ ]:
oof_preds  = oof_probs[trainval_idx].argmax(1)
oof_labels = full_df.iloc[trainval_idx]["label"].values
oof_f1     = f1_score(oof_labels, oof_preds, average="macro", zero_division=0)
log.info(f"OOF macro F1 (trainval): {oof_f1:.4f}")
print(f"OOF macro F1: {oof_f1:.4f}")

cv_f1    = np.mean(fold_scores)
sub_name = f"submission_{CFG['model_name']}_{cv_f1:.4f}.csv"
sub_df   = pd.DataFrame({
    "image_id":     submission_df["image_id"],
    "animal_class": [idx_to_class[i] for i in test_probs.argmax(1)],
})
sub_df.to_csv(SUBS / sub_name, index=False)
sub_df.to_csv(OUT  / sub_name, index=False)
np.save(LOGS / "oof_probs.npy",  oof_probs)
np.save(LOGS / "test_probs.npy", test_probs)
log.info(f"Submission: {sub_name}")
print(sub_df.head())


## 10. ZIP

In [ ]:
zip_path = OUT / f"{CFG['model_name']}_outputs.zip"
to_zip   = [*WEIGHTS.glob("*.pt"), *LOGS.glob("*"), *SUBS.glob("*.csv")]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in to_zip:
        zf.write(fp, fp.relative_to(OUT)); print(f"  + {fp.relative_to(OUT)}")
print(f"\n📦 {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)")
